# Ensemble Federated Learning with Clustering on CIFAR-10 - No Rotations, Dirichlet Label Distribution

This notebook implements the ensemble federated learning approach with client clustering **without rotation transformations** but with **Dirichlet label distribution**.

**Method Overview:**
1. Warmup phase: Clients train locally and collect weight differences
2. Clustering: Group clients based on weight difference patterns (induced by label heterogeneity)
3. Ensemble training: K specialized feature extractors (one per cluster) + shared classifier
4. Hierarchical aggregation: Within-cluster averaging for features, global averaging for classifier

**Key Characteristics:**
- No rotation-based feature heterogeneity
- **Dirichlet label distribution** (non-IID labels, controlled by α parameter)
- Clustering based on label distribution differences
- Isolates label heterogeneity impact on clustering

In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    !pip install -q torch torchvision scikit-learn matplotlib seaborn scipy
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

## Import Libraries

In [ ]:
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, confusion_matrix, classification_report
import copy
import random
import time

sys.path.append('..')
from training.ensemble_fl import EnsembleFedAvg
from training.utils import get_model, set_seed

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Configuration

## ⚠️ Memory-Saving Tips for Google Colab

**If the notebook crashes due to memory issues, try these fixes in `config.json`:**

```json
{
    "batch_size": 32,           // Reduce from 64 → saves ~50% memory per batch
    "num_clients": 25,          // Reduce from 50 → less weight storage
    "num_clusters_model": 4,    // Keep at 4 (or reduce to 2)
    "training_rounds": 20,      // Reduce from 30
    "warmup_epochs": 1          // Reduce from 2 → faster, less memory
}
```

**Why crashes happen:**
- **4 ResNet18 models** loaded simultaneously (~44MB each = 176MB)
- **Multiple clients × weight differences** stored in RAM
- **Colab Free**: 12-15GB RAM limit

**Solutions:**
1. Reduce batch size and number of clients (above)
2. Use **Colab Pro** (more RAM)
3. Run locally if possible

In [ ]:
# Load configuration from JSON
with open('config.json', 'r') as f:
    CONFIG = json.load(f)

# Add Dirichlet alpha parameter
CONFIG['dirichlet_alpha'] = 0.5  # Controls label heterogeneity (lower = more non-IID)

# Set random seeds
SEED = CONFIG['seed']
set_seed(SEED)
CONFIG['seed'] = SEED

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

# Memory monitoring
if torch.cuda.is_available():
    print(f"\nGPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB total")

## Load and Prepare CIFAR-10 Dataset

**No rotation transformations applied** - all clients use standard preprocessing.

In [ ]:
# Standard transform (no rotation)
standard_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10
print("Loading CIFAR-10 dataset...")
full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=standard_transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=standard_transform)

print(f"Full train dataset size: {len(full_train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Split train into train (80%) and validation (20%)

train_size = int(0.8 * len(full_train_dataset))
print(f"✓ All clients use identical preprocessing")

val_size = len(full_train_dataset) - train_size
print(f"\n✓ No rotation transformations applied")

print(f"  Test dataset size: {len(test_dataset)}")

train_dataset, val_dataset = torch.utils.data.random_split(print(f"  Validation dataset size: {len(val_dataset)}")

    full_train_dataset, print(f"  Train dataset size: {len(train_dataset)}")

    [train_size, val_size],print(f"\nSplit into:")

    generator=torch.Generator().manual_seed(CONFIG['seed'])
)

## Distribute Data Across Clients (Dirichlet Distribution)

**Dirichlet label distribution** - creates non-IID label heterogeneity controlled by α parameter.

- Test: IID distribution (standard practice for unbiased evaluation)
- Train/Validation: Dirichlet distribution (non-IID labels)

In [ ]:
# Organize data by class for each split
num_classes = 10

# Training data by class
train_indices_by_class = [[] for _ in range(num_classes)]
for idx in train_dataset.indices:
    _, label = full_train_dataset[idx]
    train_indices_by_class[label].append(idx)

# Validation data by class
val_indices_by_class = [[] for _ in range(num_classes)]
for idx in val_dataset.indices:
    _, label = full_train_dataset[idx]
    val_indices_by_class[label].append(idx)

# Test data by class
test_indices_by_class = [[] for _ in range(num_classes)]
for idx in range(len(test_dataset)):
    _, label = test_dataset[idx]
    test_indices_by_class[label].append(idx)

print(f"Training samples per class:")
for class_id, indices in enumerate(train_indices_by_class):
    print(f"  Class {class_id}: {len(indices)} samples")

print(f"\nValidation samples per class:")
for class_id, indices in enumerate(val_indices_by_class):
    print(f"  Class {class_id}: {len(indices)} samples")

print(f"\nTest samples per class:")
for class_id, indices in enumerate(test_indices_by_class):
    print(f"  Class {class_id}: {len(indices)} samples")

def distribute_with_dirichlet(indices_by_class, num_clients, alpha):
    """Distribute data to clients using Dirichlet distribution for non-IID label distribution."""
    client_indices = [[] for _ in range(num_clients)]
    
    for class_id in range(len(indices_by_class)):
        class_indices = indices_by_class[class_id]
        np.random.shuffle(class_indices)
        
        # Sample proportions from Dirichlet distribution
        proportions = np.random.dirichlet(alpha=[alpha] * num_clients)
        proportions = proportions / proportions.sum()  # Normalize
        
        # Distribute indices according to proportions
        split_points = (np.cumsum(proportions) * len(class_indices)).astype(int)[:-1]
        client_class_splits = np.split(class_indices, split_points)
        
        for client_id in range(num_clients):
            client_indices[client_id].extend(client_class_splits[client_id])
    
    # Shuffle each client's indices
    for client_id in range(num_clients):
        np.random.shuffle(client_indices[client_id])
    
    return client_indices

# Apply Dirichlet distribution
print(f"\nApplying Dirichlet distribution with α={CONFIG['dirichlet_alpha']}...")
client_indices_train = distribute_with_dirichlet(
    train_indices_by_class, 
    CONFIG['num_clients'], 
    CONFIG['dirichlet_alpha']
)

# Create subsets
train_subsets = [Subset(train_dataset, indices) for indices in client_indices_train]

print(f"\nCreated {len(train_subsets)} client datasets")
print(f"Average samples per client: {np.mean([len(s) for s in train_subsets]):.1f}")
print(f"Std samples per client: {np.std([len(s) for s in train_subsets]):.1f}")
print(f"Total training samples: {sum([len(s) for s in train_subsets])}")

# Calculate label distribution statistics
client_label_distributions = []
for subset in train_subsets:
    label_counts = np.zeros(num_classes)
    for idx in subset.indices:
        _, label = train_dataset[idx]
        label_counts[label] += 1
    client_label_distributions.append(label_counts)

client_label_distributions = np.array(client_label_distributions)

# Visualize data distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Samples per client
ax = axes[0, 0]
client_sizes = [len(subset) for subset in train_subsets]
ax.bar(range(CONFIG['num_clients']), client_sizes, alpha=0.7, edgecolor='black')
ax.axhline(y=np.mean(client_sizes), color='red', linestyle='--', 
           label=f'Mean: {np.mean(client_sizes):.0f}')
ax.set_xlabel('Client ID', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Samples per Client (Dirichlet Distribution)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Label distribution heatmap
ax = axes[0, 1]
im = ax.imshow(client_label_distributions.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xlabel('Client ID', fontsize=12)
ax.set_ylabel('Class Label', fontsize=12)
ax.set_title(f'Label Distribution Heatmap (α={CONFIG["dirichlet_alpha"]})', fontsize=14, fontweight='bold')
ax.set_xticks(range(0, CONFIG['num_clients'], max(1, CONFIG['num_clients']//10)))
ax.set_yticks(range(num_classes))
plt.colorbar(im, ax=ax, label='Sample Count')

# Plot 3: Classes per client
ax = axes[1, 0]
classes_per_client = [(dist > 0).sum() for dist in client_label_distributions]
ax.hist(classes_per_client, bins=range(1, num_classes + 2), alpha=0.7, edgecolor='black', color='steelblue')
ax.set_xlabel('Number of Classes', fontsize=12)
ax.set_ylabel('Number of Clients', fontsize=12)
ax.set_title('Classes per Client Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.02, 0.98, f'Mean: {np.mean(classes_per_client):.1f}\nStd: {np.std(classes_per_client):.1f}',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 4: Label entropy per client
ax = axes[1, 1]
entropies = []
for dist in client_label_distributions:
    probs = dist / dist.sum()
    probs = probs[probs > 0]  # Remove zeros

    entropy = -np.sum(probs * np.log2(probs))print(f"✓ Clustering will group clients with similar label distributions")

    entropies.append(entropy)print(f"✓ Mean classes per client: {np.mean(classes_per_client):.1f} / {num_classes}")

print(f"✓ Mean label entropy: {np.mean(entropies):.2f} bits (max: {np.log2(num_classes):.2f})")

ax.hist(entropies, bins=20, alpha=0.7, edgecolor='black', color='coral')print(f"\n✓ Dirichlet distribution (α={CONFIG['dirichlet_alpha']}) - label heterogeneity applied")

ax.axvline(x=np.mean(entropies), color='red', linestyle='--', linewidth=2,

           label=f'Mean: {np.mean(entropies):.2f}')plt.show()

ax.axvline(x=np.log2(num_classes), color='green', linestyle='--', linewidth=2,plt.savefig('ensemble_no_rotation_dirichlet_data_distribution.png', dpi=300, bbox_inches='tight')

           label=f'Max (uniform): {np.log2(num_classes):.2f}')plt.tight_layout()

ax.set_xlabel('Label Entropy (bits)', fontsize=12)

ax.set_ylabel('Number of Clients', fontsize=12)ax.grid(True, alpha=0.3, axis='y')

ax.set_title('Label Entropy Distribution', fontsize=14, fontweight='bold')ax.legend()

In [ ]:
def distribute_with_dirichlet(indices_by_class, num_clients, alpha):
    """Distribute data to clients using Dirichlet distribution for non-IID label distribution."""
    client_indices = [[] for _ in range(num_clients)]
    
    for class_id in range(len(indices_by_class)):
        class_indices = indices_by_class[class_id]
        np.random.shuffle(class_indices)
        
        # Sample proportions from Dirichlet distribution
        proportions = np.random.dirichlet(alpha=[alpha] * num_clients)
        proportions = proportions / proportions.sum()  # Normalize
        
        # Distribute indices according to proportions
        split_points = (np.cumsum(proportions) * len(class_indices)).astype(int)[:-1]
        client_class_splits = np.split(class_indices, split_points)
        
        for client_id in range(num_clients):
            client_indices[client_id].extend(client_class_splits[client_id])
    
    # Shuffle each client's indices
    for client_id in range(num_clients):
        np.random.shuffle(client_indices[client_id])
    
    return client_indices

def distribute_iid(indices_by_class, num_clients):
    """Distribute data uniformly (IID) across clients."""
    all_indices = []
    for class_indices in indices_by_class:
        all_indices.extend(class_indices)
    
    random.shuffle(all_indices)
    
    # Split uniformly
    samples_per_client = len(all_indices) // num_clients
    client_indices = []
    
    for client_id in range(num_clients):
        start_idx = client_id * samples_per_client
        end_idx = start_idx + samples_per_client if client_id < num_clients - 1 else len(all_indices)
        client_indices.append(all_indices[start_idx:end_idx])
    
    return client_indices

# Apply Dirichlet distribution to training and validation data
print(f"\nApplying Dirichlet distribution with α={CONFIG['dirichlet_alpha']}...")
client_indices_train = distribute_with_dirichlet(
    train_indices_by_class, 
    CONFIG['num_clients'], 
    CONFIG['dirichlet_alpha']
)

client_indices_val = distribute_with_dirichlet(
    val_indices_by_class, 
    CONFIG['num_clients'], 
    CONFIG['dirichlet_alpha']
)

# Apply IID distribution to test data (standard practice)
print(f"Applying IID distribution to test data...")
client_indices_test = distribute_iid(
    test_indices_by_class,
    CONFIG['num_clients']
)

# Create subsets
train_subsets = [Subset(full_train_dataset, indices) for indices in client_indices_train]
val_subsets = [Subset(full_train_dataset, indices) for indices in client_indices_val]
test_subsets = [Subset(test_dataset, indices) for indices in client_indices_test]

print(f"\n{'='*60}")
print("TRAINING DATA DISTRIBUTION")
print(f"{'='*60}")
print(f"Created {len(train_subsets)} client datasets")
print(f"Average samples per client: {np.mean([len(s) for s in train_subsets]):.1f}")
print(f"Std samples per client: {np.std([len(s) for s in train_subsets]):.1f}")
print(f"Total training samples: {sum([len(s) for s in train_subsets])}")

print(f"\n{'='*60}")
print("VALIDATION DATA DISTRIBUTION")
print(f"{'='*60}")
print(f"Average samples per client: {np.mean([len(s) for s in val_subsets]):.1f}")
print(f"Std samples per client: {np.std([len(s) for s in val_subsets]):.1f}")
print(f"Total validation samples: {sum([len(s) for s in val_subsets])}")

print(f"\n{'='*60}")
print("TEST DATA DISTRIBUTION (IID)")
print(f"{'='*60}")
print(f"Average samples per client: {np.mean([len(s) for s in test_subsets]):.1f}")
print(f"Std samples per client: {np.std([len(s) for s in test_subsets]):.1f}")
print(f"Total test samples: {sum([len(s) for s in test_subsets])}")

# Calculate label distribution statistics for training data
client_label_distributions = []
for subset in train_subsets:
    label_counts = np.zeros(num_classes)
    for idx in subset.indices:
        _, label = full_train_dataset[idx]
        label_counts[label] += 1
    client_label_distributions.append(label_counts)

client_label_distributions = np.array(client_label_distributions)

# Visualize data distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Samples per client
ax = axes[0, 0]
client_sizes = [len(subset) for subset in train_subsets]
ax.bar(range(CONFIG['num_clients']), client_sizes, alpha=0.7, edgecolor='black')
ax.axhline(y=np.mean(client_sizes), color='red', linestyle='--', 
           label=f'Mean: {np.mean(client_sizes):.0f}')
ax.set_xlabel('Client ID', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Training Samples per Client (Dirichlet Distribution)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Label distribution heatmap
ax = axes[0, 1]
im = ax.imshow(client_label_distributions.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xlabel('Client ID', fontsize=12)
ax.set_ylabel('Class Label', fontsize=12)
ax.set_title(f'Training Label Distribution Heatmap (α={CONFIG[\"dirichlet_alpha\"]})', fontsize=14, fontweight='bold')
ax.set_xticks(range(0, CONFIG['num_clients'], max(1, CONFIG['num_clients']//10)))
ax.set_yticks(range(num_classes))
plt.colorbar(im, ax=ax, label='Sample Count')

# Plot 3: Classes per client
ax = axes[1, 0]
classes_per_client = [(dist > 0).sum() for dist in client_label_distributions]
ax.hist(classes_per_client, bins=range(1, num_classes + 2), alpha=0.7, edgecolor='black', color='steelblue')
ax.set_xlabel('Number of Classes', fontsize=12)
ax.set_ylabel('Number of Clients', fontsize=12)
ax.set_title('Classes per Client Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.02, 0.98, f'Mean: {np.mean(classes_per_client):.1f}\\nStd: {np.std(classes_per_client):.1f}',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 4: Label entropy per client
ax = axes[1, 1]
entropies = []
for dist in client_label_distributions:
    probs = dist / dist.sum()
    probs = probs[probs > 0]  # Remove zeros
    entropy = -np.sum(probs * np.log2(probs))
    entropies.append(entropy)

ax.hist(entropies, bins=20, alpha=0.7, edgecolor='black', color='coral')
ax.axvline(x=np.mean(entropies), color='red', linestyle='--', linewidth=2,
           label=f'Mean: {np.mean(entropies):.2f}')
ax.axvline(x=np.log2(num_classes), color='green', linestyle='--', linewidth=2,
           label=f'Max (uniform): {np.log2(num_classes):.2f}')
ax.set_xlabel('Label Entropy (bits)', fontsize=12)
ax.set_ylabel('Number of Clients', fontsize=12)
ax.set_title('Training Label Entropy Distribution', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ensemble_no_rotation_dirichlet_data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Train/Val: Dirichlet distribution (α={CONFIG['dirichlet_alpha']}) - label heterogeneity")
print(f"✓ Test: IID distribution - unbiased evaluation")
print(f"✓ Mean label entropy: {np.mean(entropies):.2f} bits (max: {np.log2(num_classes):.2f})")
print(f"✓ Mean classes per client: {np.mean(classes_per_client):.1f} / {num_classes}")
print(f"✓ Clustering will group clients with similar label distributions")

## Create Test Dataset

In [ ]:
# Create test loader
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Test dataset size: {len(test_dataset)}")
print(f"Test batches: {len(test_loader)}")

## Initialize Ensemble FL and Run Warmup

Initialize the ensemble FL system and run warmup phase to collect weight differences from clients.

In [ ]:
print("="*70)
print("INITIALIZING ENSEMBLE FEDERATED LEARNING (NO ROTATIONS)")
print("="*70)

# Memory optimization: Clear any cached data
import gc
torch.cuda.empty_cache() if torch.cuda.is_available() else None
gc.collect()

# Check available memory
if torch.cuda.is_available():
    print(f"GPU Memory Available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.2f} GB")

# Initialize EnsembleFedAvg
ensemble_fl = EnsembleFedAvg(
    train_subsets=train_subsets,
    test_set=test_dataset,
    num_clients=CONFIG['num_clients'],
    device=device,
    model_name=CONFIG['model_name'],
    pretrained=CONFIG['pretrained'],
    num_clusters=CONFIG['num_clusters_model'],
    batch_size=CONFIG['batch_size'],
    lr=CONFIG['lr'],
    seed=CONFIG['seed']
)

print(f"\nEnsemble FL initialized:")
print(f"  Clients: {CONFIG['num_clients']}")
print(f"  Clusters: {CONFIG['num_clusters_model']}")
print(f"  Model: {CONFIG['model_name']}")

# Run warmup phase
print(f"\n{'='*70}")
print("WARMUP PHASE: Collecting Weight Differences")
print("="*70)

warmup_start = time.time()
ensemble_fl.run_warmup(
    use_fedavg=False, 
    local_epochs=CONFIG['warmup_epochs'],
    use_weight_diff=True
)
warmup_time = time.time() - warmup_start

print(f"\nWarmup completed in {warmup_time:.2f}s ({warmup_time/60:.2f} min)")
print(f"Weight differences collected from {CONFIG['num_clients']} clients")

# Clear memory after warmup
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## Perform Client Clustering

Cluster clients based on their weight difference patterns using K-Means.

**Note:** With IID data, clusters may not have strong separation since all clients see similar distributions.

In [ ]:
print(f"\n{'='*70}")
print("CLUSTERING PHASE: Grouping Clients")
print("="*70)

# Extract weight differences (FC + Layer4)
print("Extracting weight differences from FC + Layer4 layers...")
layer_grads = ensemble_fl.get_client_layer_gradients(average_across_epochs=True)

gradients = []
for client_idx in range(ensemble_fl.num_clients):
    client_grads = layer_grads[client_idx]
    selected_grads = []
    for name, grad in client_grads.items():
        if 'fc' in name or 'layer4' in name:
            selected_grads.append(grad)
    if selected_grads:
        gradients.append(np.concatenate(selected_grads))

gradient_matrix = np.array(gradients)
print(f"Weight difference matrix shape: {gradient_matrix.shape}")

# Perform K-Means clustering
print(f"\nApplying K-Means clustering (k={CONFIG['num_clusters_model']})...")
kmeans = KMeans(
    n_clusters=CONFIG['num_clusters_model'], 
    random_state=CONFIG['seed'], 
    n_init=10
)
predicted_clusters = kmeans.fit_predict(gradient_matrix)

# Calculate clustering quality metrics
silhouette = silhouette_score(gradient_matrix, predicted_clusters)

print(f"\nClustering Results:")
print(f"  Silhouette Score: {silhouette:.4f}")
print(f"  (Note: Lower scores expected with IID data - less natural separation)")
print(f"\nCluster distribution:")
for k in range(CONFIG['num_clusters_model']):
    count = np.sum(predicted_clusters == k)
    print(f"  Cluster {k}: {count} clients")

# Assign clusters to ensemble
ensemble_fl.client_clusters = {i: int(predicted_clusters[i]) for i in range(CONFIG['num_clients'])}

# Visualize clustering
plt.figure(figsize=(12, 6))

# Plot 1: Cluster distribution
plt.subplot(1, 2, 1)
cluster_counts = [np.sum(predicted_clusters == k) for k in range(CONFIG['num_clusters_model'])]
plt.bar(range(CONFIG['num_clusters_model']), cluster_counts, alpha=0.7, edgecolor='black', color='steelblue')
plt.axhline(y=CONFIG['num_clients']/CONFIG['num_clusters_model'], color='red', linestyle='--', 
            label=f'Expected: {CONFIG["num_clients"]/CONFIG["num_clusters_model"]:.1f}')
plt.xlabel('Cluster ID')
plt.ylabel('Number of Clients')
plt.title(f'Cluster Distribution\n(Silhouette: {silhouette:.4f})')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

# Plot 2: Cluster assignments
plt.subplot(1, 2, 2)
colors = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'gray']
for k in range(CONFIG['num_clusters_model']):
    mask = predicted_clusters == k
    client_ids = np.where(mask)[0]
    plt.scatter(client_ids, [k]*len(client_ids), 
               c=colors[k % len(colors)], s=50, alpha=0.6, label=f'Cluster {k}')
plt.xlabel('Client ID')
plt.ylabel('Assigned Cluster')
plt.title('Client Cluster Assignments')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yticks(range(CONFIG['num_clusters_model']))

plt.tight_layout()
plt.savefig('ensemble_no_rotation_clustering.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ With IID data, clustering may be arbitrary (no strong feature heterogeneity)")

## Initialize and Train Ensemble

Initialize the ensemble with K feature extractors and shared classifier, then train.

In [ ]:
print(f"\n{'='*70}")
print("ENSEMBLE TRAINING PHASE")
print("="*70)

# Initialize ensemble architecture
print("\nInitializing ensemble architecture...")
ensemble_fl._initialize_ensemble()
print(f"  Feature extractors: {CONFIG['num_clusters_model']} (ResNet backbones)")
print(f"  Shared classifier: 1 ({CONFIG['num_clusters_model']} × 512 → 10 classes)")

# Training configuration
num_rounds = CONFIG['training_rounds']
client_fraction = CONFIG['client_fraction']

print(f"\nTraining for {num_rounds} rounds...")
print(f"  Client fraction: {client_fraction} ({int(client_fraction * CONFIG['num_clients'])} clients/round)")
print(f"  Local epochs: 1")
print(f"  Learning rate: {CONFIG['lr']}")

# Storage for results
test_losses = []
test_accs = []
round_times = []

training_start = time.time()

for round_num in range(1, num_rounds + 1):
    round_start = time.time()
    
    # Train one round
    test_loss, test_acc = ensemble_fl.train_ensemble_round(
        round_num=round_num,
        fraction=client_fraction,
        local_epochs=1
    )
    
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    round_time = time.time() - round_start
    round_times.append(round_time)
    
    # Print progress
    if round_num % 5 == 0 or round_num == 1:
        print(f"Round {round_num}/{num_rounds} - "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}, "
              f"Time: {round_time:.2f}s")

total_training_time = time.time() - training_start

print(f"\n{'='*70}")
print("ENSEMBLE TRAINING COMPLETE")
print("="*70)
print(f"Total time (warmup + training): {warmup_time + total_training_time:.2f}s")
print(f"Training time only: {total_training_time:.2f}s ({total_training_time/60:.2f} min)")
print(f"Average time per round: {np.mean(round_times):.2f}s")
print(f"Final test accuracy: {test_accs[-1]:.4f}")
print(f"Best test accuracy: {max(test_accs):.4f} (round {np.argmax(test_accs)+1})")

## Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Loss curve
ax = axes[0]
rounds_range = range(1, num_rounds + 1)
ax.plot(rounds_range, test_losses, 'o-', linewidth=2, markersize=4, color='purple')
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Test Loss', fontsize=12)
ax.set_title('Ensemble Test Loss (No Rotations)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy curve
ax = axes[1]
ax.plot(rounds_range, test_accs, 's-', linewidth=2, markersize=4, color='green')
ax.axhline(y=max(test_accs), color='red', linestyle='--', alpha=0.5, 
           label=f'Best: {max(test_accs):.4f}')
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Ensemble Test Accuracy (No Rotations)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('ensemble_no_rotation_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training curves saved")

## Confusion Matrix and Performance Analysis

In [ ]:
import itertools

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

# Collect all predictions and true labels
all_predictions = []
all_labels = []

ensemble_model = ensemble_fl.get_ensemble_model()
ensemble_model.eval()

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = ensemble_model(inputs)
        _, predicted = outputs.max(1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_predictions)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Counts
ax = axes[0]
im1 = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im1, ax=ax)
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       xlabel='Predicted Label',
       ylabel='True Label',
       title='Confusion Matrix - Counts\nEnsemble (No Rotations)')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm[i, j], 'd'),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=9)

# Plot 2: Percentages
ax = axes[1]
im2 = ax.imshow(cm_percent, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im2, ax=ax, format='%.1f%%')
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       xlabel='Predicted Label',
       ylabel='True Label',
       title='Confusion Matrix - Percentages\nEnsemble (No Rotations)')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = cm_percent.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm_percent[i, j], '.1f'),
            ha="center", va="center",
            color="white" if cm_percent[i, j] > thresh else "black",
            fontsize=9)

plt.tight_layout()
plt.savefig('ensemble_no_rotation_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix saved")

## Detailed Performance Metrics

In [ ]:
# Classification report
print(f"\n{'='*70}")
print("CLASSIFICATION REPORT")
print(f"{'='*70}\n")
print(classification_report(all_labels, all_predictions, target_names=class_names, digits=4))

# Per-class accuracy
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(12, 6))
bars = plt.bar(class_names, per_class_accuracy, alpha=0.7, edgecolor='black', color='steelblue')
plt.axhline(y=np.mean(per_class_accuracy), color='red', linestyle='--', 
            label=f'Mean: {np.mean(per_class_accuracy):.4f}')

# Color bars based on performance
colors = ['green' if acc > 0.7 else 'orange' if acc > 0.5 else 'red' 
          for acc in per_class_accuracy]
for bar, color in zip(bars, colors):
    bar.set_color(color)
    bar.set_alpha(0.7)

plt.xlabel('Class', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Per-Class Accuracy - Ensemble (No Rotations)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.ylim([0, 1])
plt.tight_layout()
plt.savefig('ensemble_no_rotation_per_class_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPer-class accuracy statistics:")
print(f"  Mean: {np.mean(per_class_accuracy):.4f}")
print(f"  Std:  {np.std(per_class_accuracy):.4f}")
print(f"  Min:  {np.min(per_class_accuracy):.4f} ({class_names[np.argmin(per_class_accuracy)]})")
print(f"  Max:  {np.max(per_class_accuracy):.4f} ({class_names[np.argmax(per_class_accuracy)]})")

# Most confused pairs
print(f"\n{'='*70}")
print("MOST CONFUSED CLASS PAIRS")
print(f"{'='*70}\n")

confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            confused_pairs.append((class_names[i], class_names[j], cm[i, j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)

print(f"{'True Class':<15} {'Predicted As':<15} {'Count':<10} {'% of True Class':<15}")
print("-"*60)
for true_class, pred_class, count in confused_pairs[:15]:
    true_idx = class_names.index(true_class)
    total_true = cm[true_idx, :].sum()
    percentage = (count / total_true) * 100
    print(f"{true_class:<15} {pred_class:<15} {count:<10} {percentage:>6.2f}%")

## Save Results

In [ ]:
# Save results to file
results = {
    'method': 'ensemble_no_rotation_dirichlet',
    'config': CONFIG,
    'data_distribution': 'Dirichlet',
    'dirichlet_alpha': CONFIG['dirichlet_alpha'],
    'rotations_applied': False,
    'warmup_time': warmup_time,
    'training_time': total_training_time,
    'total_time': warmup_time + total_training_time,
    'num_rounds': num_rounds,
    'clustering': {
        'silhouette_score': float(silhouette),
        'cluster_distribution': {int(k): int(np.sum(predicted_clusters == k)) 
                                for k in range(CONFIG['num_clusters_model'])}
    },
    'final_test_acc': float(test_accs[-1]),
    'best_test_acc': float(max(test_accs)),
    'best_round': int(np.argmax(test_accs) + 1),
    'test_losses': [float(x) for x in test_losses],
    'test_accs': [float(x) for x in test_accs],
    'per_class_accuracy': [float(x) for x in per_class_accuracy],
    'confusion_matrix': cm.tolist(),
    'data_stats': {
        'mean_samples_per_client': float(np.mean([len(s) for s in train_subsets])),
        'std_samples_per_client': float(np.std([len(s) for s in train_subsets])),
        'mean_label_entropy': float(np.mean(entropies)),
        'std_label_entropy': float(np.std(entropies)),
        'mean_classes_per_client': float(np.mean(classes_per_client))
    }
}

with open(f'ensemble_no_rotation_alpha{CONFIG["dirichlet_alpha"]}_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to 'ensemble_no_rotation_alpha{CONFIG['dirichlet_alpha']}_results.json'")

# Save model checkpoint
torch.save({
    'round': num_rounds,
    'feature_extractors_state_dict': [fe.state_dict() for fe in ensemble_model.feature_extractors],
    'classifier_state_dict': ensemble_model.classifier.state_dict(),
    'test_acc': test_accs[-1],
    'config': CONFIG
}, f'ensemble_no_rotation_alpha{CONFIG["dirichlet_alpha"]}_checkpoint.pth')

print(f"Model checkpoint saved to 'ensemble_no_rotation_alpha{CONFIG['dirichlet_alpha']}_checkpoint.pth'")
print("\n✓ All results and visualizations saved")

## Summary

**Ensemble Federated Learning - No Rotations, Dirichlet Label Distribution:**

**Architecture:**
- K specialized feature extractors (ResNet backbones without FC layer)
- 1 shared classifier (K × 512 → 10 classes)
- Features from client's cluster models are concatenated and normalized

**Key Characteristics:**
- ✓ No rotation transformations applied (no feature heterogeneity)
- ✓ All clients use identical standard preprocessing
- ✓ **Dirichlet label distribution** (non-IID, α parameter controls heterogeneity)
- ✓ Clustering discovers clients with similar label distributions
- ✓ Isolates label heterogeneity impact on ensemble clustering

**Heterogeneity:**
- **Label heterogeneity**: Dirichlet α={CONFIG['dirichlet_alpha']}
  - Lower α → More extreme label imbalance per client
  - Higher α → More balanced label distribution
- **No feature heterogeneity**: All clients see same image transformations

**Expected Results:**
- **Meaningful clustering**: Clients with similar label distributions group together
- **Moderate silhouette score**: Clusters based on label patterns (better than IID)
- **Better than FedAvg**: Ensemble handles label heterogeneity through specialization
- **Depends on α**: Lower α → more heterogeneity → larger ensemble advantage

**Training Strategy:**
1. **Warmup**: Clients train locally, collect weight differences (Δw)
   - Label-biased clients develop distinct weight patterns
2. **Clustering**: K-Means groups clients with similar label distributions
3. **Ensemble Training**: 
   - Within-cluster: Aggregate feature extractors (similar label preferences)
   - Global: Aggregate shared classifier (learns all classes)
4. **Inference**: Concatenate features from client's cluster → shared classifier

**Comparison Goals:**
- vs **Ensemble with Rotations + Dirichlet**: Impact of adding feature heterogeneity
- vs **FedAvg (Dirichlet only)**: Ensemble advantage on label-heterogeneous data
- vs **Ensemble (IID)**: Effect of label heterogeneity on clustering quality
- vs **Centralized Training**: Total cost of federated learning with label heterogeneity

**Research Value:**
Isolates ensemble's ability to handle label heterogeneity through clustering and specialization. Shows how Dirichlet distribution creates natural client groupings that the ensemble can exploit, without confounding effects from rotation-based feature heterogeneity.

**Alpha Sensitivity:**
Run with different α values (0.1, 0.5, 1.0, 10.0) to measure clustering quality and ensemble advantage across varying levels of label heterogeneity.

**Files Generated:**
- `ensemble_no_rotation_alpha{α}_results.json` - Complete training metrics
- `ensemble_no_rotation_alpha{α}_checkpoint.pth` - Trained ensemble model
- `ensemble_no_rotation_dirichlet_data_distribution.png` - 4-panel heterogeneity visualization